#imports and setup

In [2]:
import os
import time
import random
import requests
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from PIL import Image

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


# -----------------------------
# MET API
# -----------------------------

BASE = "https://collectionapi.metmuseum.org/public/collection/v1"


# -----------------------------
# RANDOM SEEDS
# -----------------------------

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# -----------------------------
# DEVICE
# -----------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("CUDA available:", torch.cuda.is_available())
print("Using device:", device)

CUDA available: False
Using device: cpu


#api fetch functions

In [3]:
def search_objects(query, has_images=True):
    r = requests.get(
        f"{BASE}/search",
        params={
            "q": query,
            "hasImages": "true" if has_images else "false"
        },
        timeout=15
    )

    r.raise_for_status()

    return r.json().get("objectIDs", []) or []


def get_object(object_id):
    r = requests.get(
        f"{BASE}/objects/{object_id}",
        timeout=15
    )

    r.raise_for_status()

    return r.json()


def get_object_safe(object_id, max_retries=3):
    for attempt in range(max_retries):

        try:
            return get_object(object_id)

        except requests.exceptions.RequestException as e:

            if attempt == max_retries - 1:
                print(f"Failed object {object_id}: {e}")
                return None

            time.sleep(2 ** attempt)


#vessel filter

In [4]:
# vessel object names we'll accept (lowercase, partial match)
VESSEL_KEYWORDS = [
    "vase", "vessel", "amphora", "jar", "jug", "pot", "bowl",
    "krater", "kylix", "oinochoe", "hydria", "lekythos", "urn",
    "pitcher", "flask", "vial", "cup", "dish", "ewer", "olpe",
    "pyxis", "situla", "askos"
]

def is_vessel(object_name):
    if not object_name:
        return False
    name = object_name.lower()
    return any(kw in name for kw in VESSEL_KEYWORDS)



#depts

In [5]:
DEPTS = {
    13: "Greek and Roman Art",
    10: "Egyptian Art",
    3: "Ancient West Asian Art",
    5: "Arts of Africa, Oceania, and the Americas",
    6: "Asian Art"
}

print(DEPTS)

{13: 'Greek and Roman Art', 10: 'Egyptian Art', 3: 'Ancient West Asian Art', 5: 'Arts of Africa, Oceania, and the Americas', 6: 'Asian Art'}


#collect vessels by department

In [6]:
def get_department_ids(dept_id, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(
                f"{BASE}/objects",
                params={"departmentIds": dept_id},
                timeout=20
            )

            if r.status_code == 403:
                time.sleep(2 ** attempt)
                continue

            r.raise_for_status()

            return set(
                r.json().get("objectIDs", []) or []
            )

        except requests.exceptions.RequestException:
            if attempt == max_retries - 1:
                return set()

            time.sleep(2 ** attempt)

    return set()


def sample_department_vessels(dept_id, sample_size=400):
    dept_ids = get_department_ids(dept_id)

    sample_ids = random.sample(
        list(dept_ids),
        min(sample_size, len(dept_ids))
    )

    records = []

    for i, oid in enumerate(sample_ids):
        obj = get_object_safe(oid)

        if obj and is_vessel(obj.get("objectName", "")):
            records.append({
                "id": oid,
                "has_image": bool(obj.get("primaryImage")),
                "image_url": obj.get("primaryImage"),
                "culture": obj.get("culture"),
                "date_begin": obj.get("objectBeginDate"),
                "date_end": obj.get("objectEndDate"),
                "object_name": obj.get("objectName"),
                "department": dept_id
            })

        time.sleep(0.15)

        if i % 100 == 0:
            print(
                f"dept {dept_id} progress: "
                f"{i}/{len(sample_ids)}"
            )

    return records


all_vessel_records = []

SAMPLE_SIZES = {
    13: 600,   # Greek and Roman
    10: 600,   # Egyptian
    3: 1500,   # Ancient West Asian -> Near Eastern
    5: 600,
    6: 600     # East Asian
}


for dept_id in DEPTS.keys():
    print(
        f"Sampling department "
        f"{dept_id} ({DEPTS[dept_id]})..."
    )

    recs = sample_department_vessels(
        dept_id,
        sample_size=SAMPLE_SIZES[dept_id]
    )

    recs = sample_department_vessels(
        dept_id,
        sample_size=400
    )

    print(
        f"  -> {len(recs)} vessels found"
    )

    all_vessel_records.extend(recs)


full_df = pd.DataFrame(all_vessel_records)

print(
    f"\nTotal vessels across all departments: "
    f"{len(full_df)}"
)

print(
    full_df["culture"].value_counts()
)

full_df.to_csv(
    "full_df_backup.csv",
    index=False
)

print(
    "\nSaved to full_df_backup.csv"
)

Sampling department 13 (Greek and Roman Art)...
dept 13 progress: 0/600
Failed object 256675: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/256675
Failed object 249216: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/249216
Failed object 253473: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/253473
Failed object 721639: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/721639
Failed object 240471: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/240471
Failed object 721426: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/721426
Failed object 759983: 403 Client Error: Forbidden for url: https://collectionapi.metmuseum.org/public/collection/v1/objects/759983
Failed obje

#build culture map 

In [7]:
culture_map = {
    # Greek (broad bucket - includes regional styles + Cypriot)
    "Greek, Attic": "Greek",
    "Greek": "Greek",
    "Greek, Boeotian": "Greek",
    "Greek, Laconian": "Greek",
    "Greek, South Italian, Apulian": "Greek",
    "Greek, South Italian, Campanian": "Greek",
    "Greek, South Italian, Paestan": "Greek",
    "Greek, South Italian, Apulian, Canosan": "Greek",
    "East Greek/Sardis, Lydian": "Greek",
    "Cypriot": "Greek",

    # Aegean Bronze Age (Minoan/Mycenaean family)
    "Minoan": "Aegean",
    "Mycenaean": "Aegean",
    "Aegean": "Aegean",
    "Helladic": "Aegean",
    "Cycladic": "Aegean",

    # Roman
    "Roman": "Roman",
    "Roman, Gaul": "Roman",
    "Roman, Cypriot": "Roman",

    # Etruscan / Italic
    "Etruscan": "Etruscan",
    "Italic-Native, Sicilian (Centuripe)": "Etruscan",

    # Egyptian
    "Egyptian": "Egyptian",

    # Near East / Persian (broad bucket)
    "Iran": "Near Eastern",
    "Iranian": "Near Eastern",
    "Sasanian": "Near Eastern",
    "Parthian": "Near Eastern",
    "Achaemenid": "Near Eastern",
    "Sumerian": "Near Eastern",
    "Israelite": "Near Eastern",
    "Assyrian": "Near Eastern",
    "Hittite": "Near Eastern",

    # East Asian (broad bucket)
    "China": "East Asian",
    "Chinese": "East Asian",
    "Japan": "East Asian",
    "Korea": "East Asian",

    # Andean (broad bucket)
    "Paracas": "Andean",
    "Inca": "Andean",
    "Chimú": "Andean",
    "Nasca": "Andean",
    "Manteño Huancavilca": "Andean",
    "Tairona": "Andean",

    # Mesoamerican
    "Maya": "Mesoamerican",
    "Aztec": "Mesoamerican",

    # Sub-Saharan African (broad bucket, excludes Egypt)
    "Nok": "Sub-Saharan African",
    "Igbo-Ukwu": "Sub-Saharan African",
    "Yoruba": "Sub-Saharan African",
    "Benin": "Sub-Saharan African",
    "Ashanti": "Sub-Saharan African",
    "Akan": "Sub-Saharan African",
    "Luba peoples": "Sub-Saharan African",
    "Great Zimbabwe": "Sub-Saharan African",
    "Mapungubwe": "Sub-Saharan African",
    "Dogon": "Sub-Saharan African",
}

full_df["culture_clean"] = full_df["culture"].map(culture_map).fillna("Other/Unmapped")
full_df["culture_specific"] = full_df["culture"]  # keep original label for reference

print(full_df["culture_clean"].value_counts())

culture_clean
Greek                  152
Other/Unmapped          84
East Asian              60
Roman                   16
Andean                   6
Etruscan                 4
Aegean                   2
Sub-Saharan African      1
Name: count, dtype: int64


In [8]:
additional_mappings = {
    "Baule peoples": "Sub-Saharan African",
    "Bamana peoples": "Sub-Saharan African",
    "Bamum (Bamendjing)": "Sub-Saharan African",
    "Bamenda": "Sub-Saharan African",
    "Zulu peoples": "Sub-Saharan African",
    "Asante": "Sub-Saharan African",
    "Mexica (Aztec)": "Mesoamerican",
    "Veracruz": "Mesoamerican",
    "Mexican": "Mesoamerican",
    "Olmec Tradition artist, Central Highlands, Mexico": "Mesoamerican",
    "Quimbaya": "Andean",
    "Chorrera": "Andean",
    "Inca": "Andean",
    "Mogollon (Mimbres)": "Native American",
    "Pueblo": "Native American",
}

culture_map.update(additional_mappings)

full_df["culture_clean"] = full_df["culture"].map(culture_map).fillna("Other/Unmapped")
full_df.loc[full_df["department"] == 10, "culture_clean"] = "Egyptian"  # fix: Egyptian dept has blank culture field
full_df["culture_specific"] = full_df["culture"]

print(full_df["culture_clean"].value_counts())

# save so this never has to be redone
full_df.to_csv("full_df_backup.csv", index=False)
print("\nSaved.")

culture_clean
Greek                  152
East Asian              60
Egyptian                51
Other/Unmapped          28
Roman                   16
Andean                   8
Etruscan                 4
Mesoamerican             3
Aegean                   2
Sub-Saharan African      1
Name: count, dtype: int64

Saved.


#filter to solid classes

In [9]:
KEEP_CLASSES = ["Greek", "Egyptian", "East Asian", "Near Eastern", "Roman"]

train_df = full_df[
    (full_df["culture_clean"].isin(KEEP_CLASSES)) &
    (full_df["has_image"] == True)
].reset_index(drop=True)

print(train_df["culture_clean"].value_counts())
print(f"Total: {len(train_df)}")

culture_clean
Greek         143
East Asian     51
Egyptian       21
Roman          14
Name: count, dtype: int64
Total: 229


#fetch image urls

In [10]:
print(
    train_df["image_url"].isna().sum(),
    "missing URLs out of",
    len(train_df)
)


train_df = train_df[
    train_df["image_url"].notna() &
    (train_df["image_url"] != "")
].reset_index(drop=True)


print(
    "Rows remaining:",
    len(train_df)
)

0 missing URLs out of 229
Rows remaining: 229


#train val split

In [11]:
counts = train_df["culture_clean"].value_counts()

print("Class counts before split:")
print(counts)


if counts.min() < 2:
    print(
        "\nAt least one class has fewer than "
        "2 samples; using non-stratified split."
    )

    train_split, val_split = train_test_split(
        train_df,
        test_size=0.2,
        random_state=42
    )

else:
    train_split, val_split = train_test_split(
        train_df,
        test_size=0.2,
        stratify=train_df["culture_clean"],
        random_state=42
    )


train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)


print("\nTrain:")
print(
    train_split["culture_clean"].value_counts()
)

print("\nVal:")
print(
    val_split["culture_clean"].value_counts()
)


Class counts before split:
culture_clean
Greek         143
East Asian     51
Egyptian       21
Roman          14
Name: count, dtype: int64

Train:
culture_clean
Greek         114
East Asian     41
Egyptian       17
Roman          11
Name: count, dtype: int64

Val:
culture_clean
Greek         29
East Asian    10
Egyptian       4
Roman          3
Name: count, dtype: int64


#downlad images

In [12]:
import os

os.makedirs("data/images", exist_ok=True)

def download_images(df, label=""):
    failed = []
    for i, row in df.iterrows():
        fname = f"data/images/{row['id']}.jpg"
        if os.path.exists(fname):
            continue
        try:
            r = requests.get(row["image_url"], timeout=10)
            r.raise_for_status()
            with open(fname, "wb") as f:
                f.write(r.content)
        except Exception:
            failed.append(row["id"])
        time.sleep(0.2)
    print(f"{label}: {len(df) - len(failed)}/{len(df)} downloaded successfully")
    return failed

train_failed = download_images(train_split, "Train")
val_failed = download_images(val_split, "Validation")

Train: 183/183 downloaded successfully
Validation: 46/46 downloaded successfully


#remove missing images 

In [13]:
def remove_missing_images(df):
    mask = df["id"].apply(
        lambda oid:
            os.path.exists(
                f"data/images/{oid}.jpg"
            )
    )

    removed = (~mask).sum()

    print(
        f"Removing {removed} rows "
        f"with missing image files"
    )

    return df[
        mask
    ].reset_index(drop=True)


train_split = remove_missing_images(
    train_split
)

val_split = remove_missing_images(
    val_split
)

Removing 0 rows with missing image files
Removing 0 rows with missing image files


#dataset + transformations

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torch.nn as nn
from PIL import Image

CULTURE_CLASSES = sorted(train_split["culture_clean"].unique())
culture2idx = {c: i for i, c in enumerate(CULTURE_CLASSES)}
idx2culture = {i: c for c, i in culture2idx.items()}
print(culture2idx)

class PotteryDataset(Dataset):
    def __init__(self, df, culture2idx, transform=None):
        self.df = df.reset_index(drop=True)
        self.culture2idx = culture2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"data/images/{row['id']}.jpg").convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.culture2idx[row["culture_clean"]]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = PotteryDataset(train_split, culture2idx, transform=train_transform)
val_dataset = PotteryDataset(val_split, culture2idx, transform=val_transform)
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# sanity check: load one sample from the training set
img, label = train_dataset[0]
print(f"Sample image shape: {img.shape}, label: {label} ({idx2culture[label]})")

{'East Asian': 0, 'Egyptian': 1, 'Greek': 2, 'Roman': 3}
Train: 183, Val: 46
Sample image shape: torch.Size([3, 224, 224]), label: 3 (Roman)


#model definition

In [15]:
class PotteryNet(nn.Module):
    def __init__(self, n_cultures):
        super().__init__()

        backbone = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        self.features = nn.Sequential(
            *list(backbone.children())[:-1]
        )

        for p in self.features.parameters():
            p.requires_grad = False

        self.culture_head = nn.Linear(
            512,
            n_cultures
        )


    def forward(self, x):
        x = self.features(x).flatten(1)
        return self.culture_head(x)


print(
    "CUDA available:",
    torch.cuda.is_available()
)

print(
    "Using device:",
    device
)


model = PotteryNet(
    n_cultures=len(culture2idx)
).to(device)

CUDA available: False
Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 141MB/s] 


#dataloaders oversampling

In [16]:
from torch.utils.data import DataLoader, WeightedRandomSampler

# oversample based on class frequency in the training set
class_counts = train_split["culture_clean"].value_counts()
sample_weights = train_split["culture_clean"].map(lambda c: 1.0 / class_counts[c]).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

# sanity check
imgs, labels = next(iter(train_loader))
print(f"Batch shape: {imgs.shape}, labels: {labels}")

Batch shape: torch.Size([8, 3, 224, 224]), labels: tensor([0, 2, 1, 0, 3, 1, 3, 0])


#training loop

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-3
)

n_epochs = 15
patience = 3
best_val_acc = 0
epochs_no_improve = 0

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_loss/len(train_loader):.4f} - val_acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_pottery_model_cpu.pt")
        print(f"  -> new best model saved (val_acc: {val_acc:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            print(f"Best val_acc: {best_val_acc:.4f}")
            break

model.load_state_dict(
    torch.load(
        "best_pottery_model_cpu.pt",
        map_location=device
    )
)

model.eval()

print(
    "Loaded best model checkpoint."
)

Epoch 1/15 - train_loss: 1.4385 - val_acc: 0.6304
  -> new best model saved (val_acc: 0.6304)
Epoch 2/15 - train_loss: 1.1425 - val_acc: 0.7609
  -> new best model saved (val_acc: 0.7609)
Epoch 3/15 - train_loss: 0.8898 - val_acc: 0.7609
Epoch 4/15 - train_loss: 0.8242 - val_acc: 0.7391
Epoch 5/15 - train_loss: 0.7137 - val_acc: 0.8043
  -> new best model saved (val_acc: 0.8043)
Epoch 6/15 - train_loss: 0.5733 - val_acc: 0.7826
Epoch 7/15 - train_loss: 0.5525 - val_acc: 0.8043
Epoch 8/15 - train_loss: 0.5769 - val_acc: 0.7826

Early stopping triggered after 8 epochs
Best val_acc: 0.8043
Loaded best model checkpoint.


#model eval

In [18]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(
    classification_report(
        all_labels,
        all_preds,

        target_names=[
            idx2culture[i]
            for i
            in range(len(idx2culture))
        ],

        zero_division=0
    )
)

              precision    recall  f1-score   support

  East Asian       0.62      0.80      0.70        10
    Egyptian       0.50      0.50      0.50         4
       Greek       0.96      0.86      0.91        29
       Roman       0.67      0.67      0.67         3

    accuracy                           0.80        46
   macro avg       0.69      0.71      0.69        46
weighted avg       0.83      0.80      0.81        46



#single image prediction

In [19]:
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


def predict_culture(image):

    if image is None:
        return {}

    image = image.convert("RGB")

    x = inference_transform(
        image
    )

    x = x.unsqueeze(0).to(
        device
    )

    model.eval()

    with torch.no_grad():

        logits = model(x)

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]


    results = {}

    for idx, probability in enumerate(
        probabilities
    ):

        culture = idx2culture[
            idx
        ]

        results[culture] = float(
            probability.cpu().item()
        )


    return dict(
        sorted(
            results.items(),
            key=lambda item:
                item[1],
            reverse=True
        )
    )

#test single image prediction

In [20]:
test_row = val_split.iloc[0]

test_path = (
    f"data/images/"
    f"{test_row['id']}.jpg"
)

test_image = Image.open(
    test_path
)


print(
    "Actual culture:",
    test_row["culture_clean"]
)


print(
    "\nPrediction:"
)

print(
    predict_culture(
        test_image
    )
)

Actual culture: Greek

Prediction:
{'East Asian': 0.62908536195755, 'Roman': 0.16717664897441864, 'Greek': 0.16336685419082642, 'Egyptian': 0.04037116467952728}


In [21]:
print("Model classes:", culture2idx)

print("\nFull training dataframe:")
print(train_df["culture_clean"].value_counts())

print("\nTrain split:")
print(train_split["culture_clean"].value_counts())

print("\nValidation split:")
print(val_split["culture_clean"].value_counts())

Model classes: {'East Asian': 0, 'Egyptian': 1, 'Greek': 2, 'Roman': 3}

Full training dataframe:
culture_clean
Greek         143
East Asian     51
Egyptian       21
Roman          14
Name: count, dtype: int64

Train split:
culture_clean
Greek         114
East Asian     41
Egyptian       17
Roman          11
Name: count, dtype: int64

Validation split:
culture_clean
Greek         29
East Asian    10
Egyptian       4
Roman          3
Name: count, dtype: int64


In [22]:
 # ==========================================
# SAVE FINAL TRAINED MODEL
# ==========================================

torch.save(
    model.state_dict(),
    "best_pottery_model_cpu.pt"
)

print("Model saved!")

Model saved!


In [23]:
import os

print("Model saved at:")
print(os.path.abspath("best_pottery_model_cpu.pt"))

print("\nDoes it exist?")
print(os.path.exists("best_pottery_model_cpu.pt"))

Model saved at:
/content/best_pottery_model_cpu.pt

Does it exist?
True


In [24]:
from google.colab import files

files.download("/content/best_pottery_model_cpu.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
print("Model classes:", culture2idx)
print("Number of classes:", len(culture2idx))

Model classes: {'East Asian': 0, 'Egyptian': 1, 'Greek': 2, 'Roman': 3}
Number of classes: 4
